# PPO + NCAP training (Jupyter)

This notebook trains the NCAP circuit (no controller) with PPO on the `foraging` task.
Use the config cell to set the run length, then run the training cell. After training, run the video cell to render playback.

In [8]:
# Reload edited project modules on each cell run, so changes under src/ take effect
# without restarting the kernel (an already-imported module is otherwise cached).
%load_ext autoreload
%autoreload 2
# Notebook initialization: ensure `src/` is on `sys.path`, select correct kernel,
# and make `tonic` available before importing project modules.
import sys
from pathlib import Path
# Locate repository root by walking upwards until a 'src' directory is found.
cwd = Path().resolve()
repo = cwd
while repo.parent != repo and not (repo / 'src').is_dir():
    repo = repo.parent
if not (repo / 'src').is_dir():
    raise RuntimeError(f'Could not find repository root containing "src" from {cwd!s}')
src = str(repo / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
print('Inserted src at', src)
print('sys.executable:', sys.executable)
# Make tonic available (git clone if missing) and import training helpers.
from macrocircuits.tonic_setup import ensure_tonic
ensure_tonic()
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch device:', device)
if device.type == 'cuda':
    try:
        torch.set_default_device(device)
        print('Set PyTorch default device to CUDA')
    except AttributeError:
        print('torch.set_default_device not available; using CUDA-aware tensors where supported')
from macrocircuits.training import run_config, run_path, train, play_model
from macrocircuits.plotting import plot_performance

Inserted src at C:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\src
sys.executable: c:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\.venv\Scripts\python.exe
Torch device: cuda
Set PyTorch default device to CUDA


In [ ]:
# Config: adjust these values as needed
NETWORK = 'ncap'
METHOD = 'ppo'
N_LINKS = 6
TASK = 'foraging'
CONTROLLER = None  # NCAP-only; steering is learned inside the circuit (see SWIMMER_KWARGS)
STEPS = int(5e5)  # larger training like original notebook
PARALLEL = 1

# A fresh seed, on its own label. seed=0 already sits under the plain 'ncap_ppo'
# directory and showed a rising trend (last-5-epoch mean 2.08 vs first-5's 1.34,
# steering weights drifting ~0.3 from init). One seed doesn't prove that was
# learning rather than luck -- this run is the independent check. A distinct
# LABEL keeps it in its own directory so the seed=0 result isn't overwritten;
# the plot cell below loads both for a direct comparison.
SEED = 1
LABEL = f'ncap_ppo_seed{SEED}'

# Learned target-directed steering (version B): the circuit itself learns to
# turn toward food from the egocentric target vector, via a sign-free
# sensorimotor projection into the head B-neurons -- no reflex, no MLP.
#   include_target_steering=True  turns the motif on.
#   n_turn_joints=2               how many head modules the steering biases
#                                 (SMB biases head+neck; 2-3 gives more authority).
SWIMMER_KWARGS = {
    'use_weight_sharing': True,
    'include_target_steering': True,
    'n_turn_joints': 2,
}

# Pin each pellet to a fixed distance (random bearing) instead of dm_control's
# default random-box spawn. Measured on swim_to_ball: this triples the
# steering-on-vs-off effect size (41 -> 141 return) because the default spawn
# makes steering irrelevant on close targets and near-hopeless on far ones,
# diluting its benefit across episodes -- pinning the distance makes every
# episode actually require it. See envs.Swim._place_target.
TASK_KWARGS = {'target_distance': 1.0}

# Build agent/environment/trainer strings via the helper factory
agent, environment, name, trainer = run_config(
    network=NETWORK,
    method=METHOD,
    n_links=N_LINKS,
    task=TASK,
    controller=CONTROLLER,
    steps=STEPS,
    swimmer_kwargs=SWIMMER_KWARGS,
    task_kwargs=TASK_KWARGS,
    label=LABEL,
)

print('Agent:', agent)
print('Environment:', environment)
print('Experiment name:', name)
print('Trainer:', trainer)
print('Seed:', SEED)

In [ ]:
# Run training (cell will block while training runs)
train(header=None, agent=agent, environment=environment, name=name, trainer=trainer, parallel=PARALLEL, seed=SEED)

In [ ]:
# Plot reward score through training sessions
import matplotlib
import matplotlib.pyplot as plt

# Force inline notebook rendering for VS Code.
matplotlib.use('module://matplotlib_inline.backend_inline')

# Compare this seed against the original seed=0 result, to see whether the rising
# trend replicates or was specific to that one seed.
baseline_path = run_path(network=NETWORK, method=METHOD, n_links=N_LINKS, task=TASK)
this_run_path = run_path(network=NETWORK, method=METHOD, n_links=N_LINKS, task=TASK, label=LABEL)
print('Loading training logs from:', baseline_path, 'and', this_run_path)

fig, ax = plt.subplots(figsize=(10, 5))
plot_performance(
    [baseline_path, this_run_path],
    ax=ax,
    x='episodes',  # count episodes trained on, not environment steps
    title=f'Reward score through training: seed 0 vs seed {SEED}',
)
fig.savefig('reward_training_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Create a video from the trained model
play_model(run_path(network=NETWORK, method=METHOD, n_links=N_LINKS, task=TASK, label=LABEL), checkpoint='last')